# dbt-for-apache-doris

## 5 个端到端数据工程 Demo

从 Doris Source 开始，依次运行 dbt model、Data Test 和 verifier。第一个 Demo 按数据转换过程逐步执行，后四个 Demo 保留一键运行。每个 Demo 使用独立 Database，可以单独重跑；完整原始日志默认折叠，排错时再展开。

| Demo | 主要能力 | 最终对象 |
| --- | --- | --- |
| 每日订单汇总 | Table、Data Test、分区分桶、Async MV | 每日和月度收入 |
| 客户地域分析 | 跨 Database Source、View、`ref()` | 州级客户和收入指标 |
| 广告数据合并 | Seed、`dbt_utils`、`QUALIFY` | 三渠道统一明细 |
| 迟到订单 | Incremental `merge`、Unique Key | 去重后的订单当前版本 |
| 客户 Snapshot | SCD Type 2、Hard Delete | 客户历史和当前维表 |

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
from pathlib import Path
import sys


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "extension/dbt-doris/examples/data-eng-bench-doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 Apache Doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
sys.path.insert(0, str(demo_dir / "scripts"))
from notebook_helpers import DemoRunner

runner = DemoRunner()
runner.show_environment()

## 2. Demo 1：每日订单汇总

这个 Demo 拆成 6 个可单独运行的步骤。每一步先展示输入或 dbt 文件，再执行命令并查询 Doris 结果。

```text
6 条源订单
  └─ 过滤 CANCELLED / RETURNED / FAILED
       └─ 3 条有效订单
            └─ 按日期聚合：daily_order_summary（3 行）
                 └─ 按月份聚合：monthly_order_summary_mv（1 行）
```

### 2.1 准备并查看源订单

Fixture 创建 6 条订单。最后一列直接标出每条记录会进入模型还是会被过滤。

In [ ]:
daily_demo_dir = runner.examples_root / "data-eng-bench-daily-order-summary"
runner.show_file("Fixture SQL", daily_demo_dir / "scripts/setup.sql")
runner.run_sql_file("创建源订单", daily_demo_dir / "scripts/setup.sql")
runner.query("输入：6 条原始订单", """
select
    order_id, ordered_at, grand_total, status,
    case
        when status in ('CANCELLED', 'RETURNED', 'FAILED') then '过滤'
        else '进入每日汇总'
    end as transform_action
from dbt_demo_daily_source.orders
order by order_id
""")

### 2.2 将 Doris 表声明为 dbt Source

`sources.yml` 把 `source('orders', 'orders')` 映射到 Doris 的 `dbt_demo_daily_source.orders`。`dbt debug` 随后检查这套 profile 能否连接 Doris。

In [ ]:
runner.show_file("dbt Source 配置", daily_demo_dir / "models/staging/sources.yml")
runner.run_dbt("检查 Demo 的 Doris 连接", daily_demo_dir, "debug")

### 2.3 执行每日汇总 Model

Model 读取 Source，过滤 3 种无效状态，然后按 `order_date` 聚合。dbt-doris 根据 `config()` 创建带 Range Partition、Duplicate Key 和 Hash Bucket 的 Doris Table。

In [ ]:
runner.show_file("每日汇总 Model", daily_demo_dir / "models/marts/daily_order_summary.sql")
runner.run_dbt("创建 daily_order_summary", daily_demo_dir, "run", "--select", "daily_order_summary")
runner.query("输出：3 天的有效订单汇总", """
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date
""")

### 2.4 执行 Data Test

这一步不再转换数据，而是检查每日结果：日期必须非空且唯一，订单数和收入必须非空。

In [ ]:
runner.show_file("Data Test 定义", daily_demo_dir / "models/marts/daily_order_summary.yml")
runner.run_dbt("验证 daily_order_summary", daily_demo_dir, "test", "--select", "daily_order_summary")

### 2.5 从每日表生成月度异步物化视图

第二个 Model 不再读取源订单，而是通过 `ref('daily_order_summary')` 读取上一步的 3 行结果，再按月份聚合。

In [ ]:
runner.show_file("月度物化视图 Model", daily_demo_dir / "models/marts/monthly_order_summary_mv.sql")
runner.run_dbt("创建月度异步物化视图", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.run_dbt("提交月度物化视图刷新", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.query("Doris 物化视图任务", """
select MvName, Status, CreateTime
from tasks('type'='mv')
where MvDatabaseName = 'dbt_demo_daily'
  and MvName = 'monthly_order_summary_mv'
order by CreateTime desc
limit 1
""")

### 2.6 校验完整数据链并查看最终结果

Verifier 等待异步刷新完成，同时检查每日数据、Table 的 Key/Partition/Bucket DDL，以及月度汇总值。

In [ ]:
runner.run_script("校验每日订单完整数据链", daily_demo_dir / "scripts/verify.sh")
runner.query("最终结果：月度订单汇总", """
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv
order by order_month
""")

## 3. Demo 2：客户地域分析

**执行链**：地址 Source + 订单 Source → 2 个 staging View → `fct_state_customers` Table

通过 `source()` 和 `ref()` 读取两个 Doris Database，输出州级客户数、订单数和收入。

In [ ]:
runner.run_demo("客户地域分析", "data-eng-bench-doris-demos/geographic")
runner.query("州级客户与收入", """
select state_province, customer_count, order_count, total_revenue, avg_order_value
from dbt_demo_geographic.fct_state_customers
order by state_province
""")

## 4. Demo 3：广告数据标准化和合并

**执行链**：3 个 CSV → 3 个 Seed Table → 3 个去重 View → `int__ads_unified` Table

统一 Google、Meta 和 TikTok 字段，并用 `dbt_utils` 检查 `source + ad_date` 唯一性。

In [ ]:
runner.run_demo("广告数据标准化和合并", "data-eng-bench-doris-demos/consolidate")
runner.query("统一广告明细", """
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date
""")

## 5. Demo 4：迟到订单 Incremental

**执行链**：订单事件 → 版本历史 Table → Incremental Unique Key Table → 每日汇总

先全量构建，再写入订单 101 的新版本和新订单 104，执行 `merge`，最后验证无输入变化时结果不漂移。

In [ ]:
runner.run_demo("迟到订单 Incremental", "data-eng-bench-doris-demos/incremental")
runner.query("订单当前版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")
runner.query("每日销售汇总", """
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date
""")

## 6. Demo 5：客户 Snapshot

**执行链**：当前客户 Source → staging View → SCD Type 2 Snapshot → 当前客户维表

第二轮修改客户 1 并删除客户 2，验证旧版本关闭、新版本生成和 Hard Delete 失效。

In [ ]:
runner.run_demo("客户 Snapshot", "data-eng-bench-doris-demos/snapshot")
runner.query("客户历史版本", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")
runner.query("当前客户维表", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

## 完成

Demo 1 的 6 个步骤和后面 4 个 Demo 全部显示绿色“运行通过”，表示对应 dbt node、Doris 对象和 verifier 均已通过。需要排查编译 SQL 或执行细节时，展开每个结果下方的“查看完整运行日志”。